In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "data" / "creditcard.csv").exists()
        and (path / "results" / "split_indices.npz").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate project data and results/split_indices.npz. "
        "Run Notebook 01 first."
    )

df = pd.read_csv(PROJECT_ROOT / "data" / "creditcard.csv")
with np.load(PROJECT_ROOT / "results" / "split_indices.npz") as split_indices:
    train_idx = split_indices["train_idx"]
    validation_idx = split_indices["validation_idx"]

train_df = df.loc[train_idx].copy()
validation_df = df.loc[validation_idx].copy()


In [4]:


import json
from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)

SEED = 42

# Use the original dataset's feature list so training-only exploratory columns
# are not accidentally included.
feature_cols = [col for col in df.columns if col != "Class"]

X_train = train_df[feature_cols]
y_train = train_df["Class"]
X_val = validation_df[feature_cols]
y_val = validation_df["Class"]

# Keep the initial search small. Select settings by validation average precision.
candidate_params = [
    {
        "n_estimators": 100,
        "max_depth": 16,
        "min_samples_leaf": 2,
        "class_weight": "balanced",
    },
    {
        "n_estimators": 100,
        "max_depth": 16,
        "min_samples_leaf": 2,
        "class_weight": "balanced_subsample",
    },
]

best_model = None
best_params = None
best_average_precision = -np.inf
candidate_results = []

for candidate_number, params in enumerate(candidate_params):
    print(f"Training candidate {candidate_number}/{len(candidate_params)}", flush=True)
    
    model = RandomForestClassifier(
        **params,
        random_state=SEED,
        n_jobs=-1,
        verbose=1
    )
    model.fit(X_train, y_train)

    fraud_index = np.flatnonzero(model.classes_ == 1)
    if len(fraud_index) != 1:
        raise ValueError("Expected the training labels to contain fraud class 1.")

    val_scores = model.predict_proba(X_val)[:, fraud_index[0]]
    average_precision = average_precision_score(y_val, val_scores)

    candidate_results.append({
        "parameters": params,
        "validation_average_precision": float(average_precision),
    })

    if average_precision > best_average_precision:
        best_model = model
        best_params = params
        best_average_precision = average_precision

if best_model is None or best_params is None:
    raise RuntimeError("No Random Forest candidate was successfully fitted.")

# Select the operating threshold for the chosen model using validation labels.
val_scores = best_model.predict_proba(X_val)[:, np.flatnonzero(best_model.classes_ == 1)[0]]
precision, recall, thresholds = precision_recall_curve(y_val, val_scores)

if len(thresholds) == 0:
    raise ValueError("No validation thresholds available.")

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)
best_threshold_index = int(np.argmax(f1_scores))
threshold = float(thresholds[best_threshold_index])
val_predictions = (val_scores >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_val,
    val_predictions,
    labels=[0, 1],
).ravel()

validation_metrics = {
    "threshold": threshold,
    "threshold_selection": "Maximum validation F1",
    "precision": float(precision_score(y_val, val_predictions, zero_division=0)),
    "recall": float(recall_score(y_val, val_predictions, zero_division=0)),
    "f1": float(f1_score(y_val, val_predictions, zero_division=0)),
    "average_precision": float(average_precision_score(y_val, val_scores)),
    "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
}

print("Selected parameters:", best_params)
print(f"Selected validation threshold: {threshold:.6f}")
print("Validation metrics:")
print(json.dumps(validation_metrics, indent=2))

# Save the selected model and the validation-based decisions.
project_root = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "data" / "creditcard.csv").exists()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/creditcard.csv."
    )

results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, results_dir / "random_forest.joblib")

record = {
    "model_type": "Supervised class-weighted Random Forest",
    "training_label_access": (
        "The model was trained using fraud labels; unlike the unsupervised detectors, "
        "it is a supervised benchmark."
    ),
    "random_seed": SEED,
    "feature_cols": feature_cols,
    "model_selection_metric": "validation average precision",
    "selected_parameters": best_params,
    "candidate_results": candidate_results,
    "validation_metrics": validation_metrics,
}

with (results_dir / "random_forest_config.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(record, file, indent=2)

print(f"Saved model and validation record to {results_dir}")

Training candidate 0/2


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   46.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.5min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.3s


Training candidate 1/2


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.2min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.3s finished


Selected parameters: {'n_estimators': 100, 'max_depth': 16, 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample'}
Selected validation threshold: 0.549586
Validation metrics:
{
  "threshold": 0.549585516420921,
  "threshold_selection": "Maximum validation F1",
  "precision": 0.9814814814814815,
  "recall": 0.7162162162162162,
  "f1": 0.828125,
  "average_precision": 0.810990749809324,
  "false_positive_rate": 2.344830820456304e-05,
  "true_negatives": 42646,
  "false_positives": 1,
  "false_negatives": 21,
  "true_positives": 53
}
Saved model and validation record to c:\Users\user\Documents\AlgomaU courses\Computer Science\Third Year\Data Science with Python\Data Science Final Project\results
